# Generate spectrograms from classifier-conditioned VAE

In [1]:
# !pip install -r requirements.txt

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

NameError: name 'torch' is not defined

In [ ]:
#import matplotlib.pyplot as plt
import torch

from nsynth_pipeline.convexnet import freeze_module, load_convnext_backbone
from nsynth_pipeline.data import MelConfig, NOTE_NAMES, make_nsynth_loaders
from nsynth_pipeline.models import ConditionalSpectrogramVAE, ConvNeXtNoteClassifier
from nsynth_pipeline.train_utils import move_batch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

ModuleNotFoundError: No module named 'nsynth_pipeline'

In [ ]:
DATASET_NAME = "jg583/NSynth"
BATCH_SIZE = 8
NUM_WORKERS = 2

CLASSIFIER_CHECKPOINT = "checkpoints/convnext_pitch_classifier.pt"
VAE_CHECKPOINT = "checkpoints/conditional_spectrogram_vae.pt"


mel_config = MelConfig()

In [ ]:
# In Colab, this stores the Hugging Face dataset cache on Google Drive so it persists across runtimes.
HF_CACHE_DIR = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    HF_CACHE_DIR = "/content/drive/MyDrive/hf_datasets_cache"
except ModuleNotFoundError:
    print("Not running in Colab; using the local Hugging Face cache.")

print("HF cache dir:", HF_CACHE_DIR)

In [ ]:
loaders = make_nsynth_loaders(
    dataset_name=DATASET_NAME,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    max_train_items=128,
    max_test_items=128,
    mel_config=mel_config,
    cache_dir=HF_CACHE_DIR,
)
batch = move_batch(next(iter(loaders["test"])), device)
input_shape = tuple(batch["spectrogram"].shape[1:])

In [ ]:
backbone = load_convnext_backbone()
freeze_module(backbone)
classifier = ConvNeXtNoteClassifier(backbone).to(device)
classifier_checkpoint = torch.load(CLASSIFIER_CHECKPOINT, map_location=device)
classifier.load_state_dict(classifier_checkpoint["model_state_dict"])
classifier.eval()

vae_checkpoint = torch.load(VAE_CHECKPOINT, map_location=device)
vae = ConditionalSpectrogramVAE(
    input_shape=tuple(vae_checkpoint.get("input_shape", input_shape)),
    latent_dim=int(vae_checkpoint.get("latent_dim", 128)),
).to(device)
vae.load_state_dict(vae_checkpoint["model_state_dict"])
vae.eval()

In [ ]:
with torch.no_grad():
    outputs = classifier(batch["spectrogram"])
    predicted_note = outputs["note_logits"].argmax(dim=-1)
    predicted_pitch = outputs["pitch_logits"].argmax(dim=-1)
    generated = vae.generate(predicted_pitch, device=device).cpu()

[(NOTE_NAMES[i], NOTE_NAMES[j], int(k)) for i, j, k in zip(batch["note_class"].cpu().tolist(), predicted_note.cpu().tolist(), predicted_pitch.cpu().tolist())]

In [ ]:
count = min(4, generated.shape[0])
fig, axes = plt.subplots(count, 2, figsize=(10, 3 * count), squeeze=False)
for row in range(count):
    axes[row, 0].imshow(batch["spectrogram"][row, 0].detach().cpu(), aspect="auto", origin="lower")
    axes[row, 0].set_title(f"Ground truth {NOTE_NAMES[int(batch['note_class'][row])]}")
    axes[row, 1].imshow(generated[row, 0], aspect="auto", origin="lower")
    axes[row, 1].set_title(f"Generated from pitch {int(predicted_pitch[row])} / {NOTE_NAMES[int(predicted_note[row])]}")
    for axis in axes[row]:
        axis.axis("off")
plt.tight_layout()